<a href="https://www.kaggle.com/code/nihalabhay/chest-imagenet?scriptVersionId=343436361" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# ===== STAGE 14 v3, FULL CELL: EFFICIENTNETB0 PARTIAL UNFREEZE, CONTIGUOUS TAIL FROM block6d =====
!pip install -q tensorflow==2.19.0

import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import random, gc
import numpy as np
import pandas as pd
import glob
import tensorflow as tf
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print(f"Seed {SEED} set, TF {tf.__version__}, tf.keras module: {tf.keras.__name__}")
assert 'tf_keras' in tf.keras.__name__, "STOP: Keras 3 active, not legacy. Restart the session before continuing."

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_pre
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger
from tensorflow.keras.metrics import AUC
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, confusion_matrix

CHEX_DIR      = '/kaggle/input/datasets/ashery/chexpert/'
CHEX_CSV      = '/kaggle/input/datasets/ashery/chexpert/train.csv'
NIH_DIR       = '/kaggle/input/datasets/organizations/nih-chest-xrays/data/'
NIH_CSV       = '/kaggle/input/datasets/organizations/nih-chest-xrays/data/Data_Entry_2017.csv'
VINBIG_CSV    = '/kaggle/input/competitions/vinbigdata-chest-xray-abnormalities-detection/train.csv'
VINBIG_PNG    = '/kaggle/input/datasets/xhlulu/vinbigdata-chest-xray-png-512px-original-ratio/train/'

# If you persisted Phase 1 to a dataset, point this at it to skip ~2h of retraining.
# Leave as None to train Phase 1 fresh.
PHASE1_CHECKPOINT = None   # e.g. '/kaggle/input/datasets/nihalabhay/best-models/s14_partial_phase1.keras'

FINAL_CLASSES = ['no_finding', 'pathology']
IMG_SIZE, BATCH_SIZE = 224, 32
SUBSAMPLE_SEED = 42
TARGET_PER_SOURCE = 15000
PHASE1_EPOCHS, PHASE1_LR, PHASE2_LR, EARLYSTOP_PAT = 10, 1e-3, 1e-5, 7
AUG = dict(rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
           horizontal_flip=True, zoom_range=0.1)

chex = pd.read_csv(CHEX_CSV)
chex['label'] = np.where(chex['No Finding'] == 1.0, 'no_finding', 'pathology')
chex['patient_id'] = chex['Path'].str.extract(r'(patient\d+)')
chex['image_path'] = CHEX_DIR + chex['Path'].str.replace('CheXpert-v1.0-small/', '', regex=False)
chex['source'] = 'chex'

nih = pd.read_csv(NIH_CSV)
nih['label'] = np.where(nih['Finding Labels'] == 'No Finding', 'no_finding', 'pathology')
nih['patient_id'] = nih['Patient ID'].astype(str)
nih['source'] = 'nih'
nih_files = glob.glob(os.path.join(NIH_DIR, '**', '*.png'), recursive=True)
nih_map = {os.path.basename(p): p for p in nih_files}
nih['image_path'] = nih['Image Index'].map(nih_map)

vin_raw = pd.read_csv(VINBIG_CSV)
img_findings = vin_raw.groupby('image_id')['class_id'].apply(lambda s: set(s))
vin = pd.DataFrame({'image_id': img_findings.index})
vin['label'] = img_findings.apply(lambda fs: 'no_finding' if fs == {14} else 'pathology').values
vin['image_path'] = VINBIG_PNG + vin['image_id'] + '.png'
vin['source'] = 'vinbig'; vin['patient_id'] = None

def subsample_by_patient(df, target_n, seed=SUBSAMPLE_SEED):
    pats = df['patient_id'].drop_duplicates().sample(frac=1.0, random_state=seed).tolist()
    sizes = df['patient_id'].value_counts()
    chosen, count = [], 0
    for p in pats:
        n = sizes[p]
        if count + n > target_n and count >= target_n * 0.98: break
        chosen.append(p); count += n
        if count >= target_n: break
    return df[df['patient_id'].isin(chosen)].reset_index(drop=True)

chex_s = subsample_by_patient(chex, TARGET_PER_SOURCE)
nih_s  = subsample_by_patient(nih,  TARGET_PER_SOURCE)
vin_s  = vin.copy()

def safe_split(df, label_col, test_size, rs, tag=""):
    try:
        return train_test_split(df, test_size=test_size, stratify=df[label_col], random_state=rs)
    except ValueError as e:
        print(f"WARNING [{tag}]: stratified split failed, unstratified fallback. {e}")
        return train_test_split(df, test_size=test_size, random_state=rs)

def split_patient_level(df, rs=SEED, tag=""):
    pg = df.groupby('patient_id')['label'].agg(lambda s: s.value_counts().index[0]).reset_index()
    p_tr, p_tmp = safe_split(pg, 'label', 0.30, rs, f"{tag} first")
    p_va, p_te  = safe_split(p_tmp, 'label', 0.50, rs, f"{tag} second")
    pick = lambda ids: df[df['patient_id'].isin(ids['patient_id'])]
    return pick(p_tr), pick(p_va), pick(p_te)

def split_image_level(df, rs=SEED, tag="VinBig"):
    tr, tmp = safe_split(df, 'label', 0.30, rs, f"{tag} first")
    va, te  = safe_split(tmp, 'label', 0.50, rs, f"{tag} second")
    return tr, va, te

c_tr, c_va, c_te = split_patient_level(chex_s, tag="CheXpert")
n_tr, n_va, n_te = split_patient_level(nih_s,  tag="NIH")
v_tr, v_va, v_te = split_image_level(vin_s)

train_df = pd.concat([c_tr, n_tr, v_tr], ignore_index=True)
val_df   = pd.concat([c_va, n_va, v_va], ignore_index=True)
test_df  = pd.concat([c_te, n_te, v_te], ignore_index=True)
print(f"Pooled Train {len(train_df):,} | Val {len(val_df):,} | Test {len(test_df):,}")

cls = np.array(FINAL_CLASSES)
cw  = compute_class_weight('balanced', classes=cls, y=train_df['label'])
CLASS_WEIGHT = {i: w for i, w in enumerate(cw)}

def make_gens(preprocess_fn):
    train_idg = ImageDataGenerator(preprocessing_function=preprocess_fn, **AUG)
    eval_idg  = ImageDataGenerator(preprocessing_function=preprocess_fn)
    common = dict(x_col='image_path', y_col='label', target_size=(IMG_SIZE,IMG_SIZE),
                  batch_size=BATCH_SIZE, class_mode='categorical', classes=FINAL_CLASSES, color_mode='rgb')
    tr = train_idg.flow_from_dataframe(train_df, shuffle=True, seed=SEED, **common)
    va = eval_idg.flow_from_dataframe(val_df, shuffle=False, **common)
    te = eval_idg.flow_from_dataframe(test_df, shuffle=False, **common)
    return tr, va, te

tr, va, te = make_gens(eff_pre)
print("class_indices:", te.class_indices)

# ---------- Phase 1: fully frozen backbone ----------
if PHASE1_CHECKPOINT and os.path.exists(PHASE1_CHECKPOINT):
    print(f"Loading persisted Phase 1 checkpoint: {PHASE1_CHECKPOINT}")
    model = load_model(PHASE1_CHECKPOINT)
else:
    print("Training Phase 1 fresh (frozen backbone, 10 epochs)")
    base = EfficientNetB0(include_top=False, weights='imagenet', input_shape=(IMG_SIZE, IMG_SIZE, 3))
    model = Sequential([base, GlobalAveragePooling2D(), Dense(256, activation='relu'),
                        Dropout(0.3), Dense(2, activation='softmax')])
    base.trainable = False
    model.compile(Adam(PHASE1_LR), 'categorical_crossentropy', ['accuracy', AUC(name='auc')])
    model.fit(tr, validation_data=va, epochs=PHASE1_EPOCHS, class_weight=CLASS_WEIGHT,
              callbacks=[ModelCheckpoint('/kaggle/working/s14_partial_phase1.keras',
                                          monitor='val_auc', mode='max', save_best_only=True),
                         CSVLogger('/kaggle/working/s14_partial_phase1_log.csv', append=False)],
              verbose=1)
    print("Phase 1 (frozen) done. DOWNLOAD s14_partial_phase1.keras NOW so a rerun can skip it.")
    del model, base; gc.collect(); tf.keras.backend.clear_session()
    model = load_model('/kaggle/working/s14_partial_phase1.keras')

# ---------- Phase 2: contiguous tail unfreeze from block6d to end of backbone ----------
# Unfreezing a CONTIGUOUS TAIL by index (not by block-name matching) is the actual fix.
# Name-matching left top_conv/top_bn/top_activation frozen while gradients still had to flow
# back through them to reach block6d/block7a. A frozen BatchNorm runs in inference mode, and
# TF has no deterministic GPU kernel for fused BN backprop in inference mode, which is what
# crashed at top_bn twice. A contiguous tail means every layer in the backward path is trainable.
# It also makes the ablation cleaner: frozen / partial / full now differ ONLY in unfreeze depth,
# with BatchNorm handled identically to Stage 5's full unfreeze.
base = model.layers[0]
UNFREEZE_FROM_BLOCK = 'block6d'

first_idx = next(i for i, l in enumerate(base.layers) if l.name.startswith(UNFREEZE_FROM_BLOCK))
base.trainable = True
for i, layer in enumerate(base.layers):
    layer.trainable = (i >= first_idx)

trainable_layers = [l.name for l in base.layers if l.trainable]
frozen_bn_in_path = [l.name for i, l in enumerate(base.layers)
                     if i >= first_idx and 'BatchNormalization' in l.__class__.__name__ and not l.trainable]
print(f"Unfreeze starts at index {first_idx} ({base.layers[first_idx].name})")
print(f"Trainable backbone layers: {len(trainable_layers)} of {len(base.layers)}")
print(f"Tail includes: {trainable_layers[-3:]}")
print(f"Frozen BatchNorm layers in the gradient path (MUST be empty): {frozen_bn_in_path}")
assert len(frozen_bn_in_path) == 0, "A frozen BatchNorm sits in the backward path, this will crash. Do not proceed."

model.compile(Adam(PHASE2_LR), 'categorical_crossentropy', ['accuracy', AUC(name='auc')])
model.fit(tr, validation_data=va, epochs=60, class_weight=CLASS_WEIGHT,
          callbacks=[EarlyStopping(monitor='val_auc', mode='max', patience=EARLYSTOP_PAT, restore_best_weights=True),
                     ModelCheckpoint('/kaggle/working/s14_partial_best.keras', monitor='val_auc', mode='max', save_best_only=True),
                     CSVLogger('/kaggle/working/s14_partial_phase2_log.csv', append=False)],
          verbose=1)

preds = model.predict(te, verbose=0)
proba = preds[:,1]; y = np.array(te.classes); yhat = (proba >= 0.5).astype(int)
auc = roc_auc_score(y, proba); acc = accuracy_score(y, yhat); f1 = f1_score(y, yhat, average='macro')
tn, fp, fn, tp = confusion_matrix(y, yhat).ravel()
sens = tp/(tp+fn); spec = tn/(tn+fp) if (tn+fp) > 0 else float('nan')
print(f"\nSTAGE 14 PARTIAL UNFREEZE TEST: AUC={auc:.4f} Acc={acc:.4f} MacroF1={f1:.4f} Sens={sens:.4f} Spec={spec:.4f}")
print("Confusion [tn,fp,fn,tp]:", tn, fp, fn, tp)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.0/645.0 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 104.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
ydf-tf 2.20.0 requires tensorflow==2.20.0, but you have tensorflow 2.19.0 which is incompatible.
tf-keras 2.20.0 requires tensorflow<2.21,>=2.20, but you have tensorflow 2.19.0 which is incompatible.
tensorflow-text 2.20.1 requires tensorflow<2.21,>=2.20.0, but you have tensorflow 2.19.0 which is incompatible.


2026-08-19 10:02:42.885469: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1787133762.909375      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1787133762.923701      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1787133762.948647      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1787133762.948677      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1787133762.948680      23 computation_placer.cc:177] computation placer alr

Seed 42 set, TF 2.19.0, tf.keras module: tf_keras.api._v2.keras
Pooled Train 31,456 | Val 7,015 | Test 6,505
Found 31456 validated image filenames belonging to 2 classes.
Found 7015 validated image filenames belonging to 2 classes.
Found 6505 validated image filenames belonging to 2 classes.
class_indices: {'no_finding': 0, 'pathology': 1}
Training Phase 1 fresh (frozen backbone, 10 epochs)


I0000 00:00:1787134096.940948      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1787134096.947518      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


16705208/16705208 [==============================] - 0s 0us/step
Epoch 1/10


E0000 00:00:1787134106.773636      23 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape insequential/efficientnetb0/block2b_drop/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
I0000 00:00:1787134108.172793      76 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1787134110.422220      74 service.cc:152] XLA service 0x7a3acd2e20b0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1787134110.422269      74 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1787134110.422277      74 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1787134110.582751      74 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


983/983 [==============================] - 978s 986ms/step - loss: 0.4816 - accuracy: 0.7662 - auc: 0.8491 - val_loss: 0.4436 - val_accuracy: 0.7926 - val_auc: 0.8745
Epoch 2/10
983/983 [==============================] - 729s 741ms/step - loss: 0.4457 - accuracy: 0.7866 - auc: 0.8725 - val_loss: 0.4296 - val_accuracy: 0.7919 - val_auc: 0.8817
Epoch 3/10
983/983 [==============================] - 727s 740ms/step - loss: 0.4354 - accuracy: 0.7946 - auc: 0.8789 - val_loss: 0.4272 - val_accuracy: 0.7950 - val_auc: 0.8838
Epoch 4/10
983/983 [==============================] - 761s 774ms/step - loss: 0.4252 - accuracy: 0.7998 - auc: 0.8849 - val_loss: 0.4161 - val_accuracy: 0.8108 - val_auc: 0.8922
Epoch 5/10
983/983 [==============================] - 735s 747ms/step - loss: 0.4197 - accuracy: 0.8036 - auc: 0.8884 - val_loss: 0.4038 - val_accuracy: 0.8093 - val_auc: 0.8968
Epoch 6/10
983/983 [==============================] - 731s 744ms/step - loss: 0.4138 - accuracy: 0.8067 - auc: 0.8918 - v

E0000 00:00:1787141786.158313      23 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape insequential/efficientnetb0/block2b_drop/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


983/983 [==============================] - 773s 774ms/step - loss: 0.4862 - accuracy: 0.7730 - auc: 0.8549 - val_loss: 0.4231 - val_accuracy: 0.7991 - val_auc: 0.8857
Epoch 2/60
983/983 [==============================] - 762s 775ms/step - loss: 0.4434 - accuracy: 0.7899 - auc: 0.8762 - val_loss: 0.4138 - val_accuracy: 0.8047 - val_auc: 0.8910
Epoch 3/60
983/983 [==============================] - 752s 765ms/step - loss: 0.4243 - accuracy: 0.8019 - auc: 0.8865 - val_loss: 0.4068 - val_accuracy: 0.8091 - val_auc: 0.8947
Epoch 4/60
983/983 [==============================] - 764s 777ms/step - loss: 0.4161 - accuracy: 0.8052 - auc: 0.8908 - val_loss: 0.3999 - val_accuracy: 0.8128 - val_auc: 0.8983
Epoch 5/60
983/983 [==============================] - 786s 800ms/step - loss: 0.4063 - accuracy: 0.8112 - auc: 0.8962 - val_loss: 0.3979 - val_accuracy: 0.8133 - val_auc: 0.8993
Epoch 6/60
983/983 [==============================] - 770s 783ms/step - loss: 0.4005 - accuracy: 0.8139 - auc: 0.8992 - v